# Real Data Feature Builder v3 (GPU Batch Path)

方案1（低侵入）实现：
1. 整条信号只预处理一次。
2. 滑窗后按批处理窗口。
3. 三个 `SC_mean` 走 GPU 批量 FFT（一次大调用）。
4. `C_f`、`C_h` 仅保留 `b_1k_10k` 的 context 计算。


In [9]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import replace
from datetime import datetime, timedelta
from pathlib import Path
import logging
import os
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

os.environ.setdefault('FEA_CPT_USE_GPU', '1')

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt_gpu.base import FeatureRecord
from fea_cpt_gpu.params import DEFAULT_FEATURE_PARAMS
from fea_cpt_gpu.signal_ops import build_context, butter_filter
from fea_cpt_gpu.gpu_backend import gpu_backend_info

try:
    from nptdms import TdmsFile
except ImportError:
    TdmsFile = None

print(f'workspace = {workspace}')
print(gpu_backend_info())
print('nptdms =', 'available' if TdmsFile is not None else 'missing')


workspace = e:\codes\ZZ-BK
[GPU] 使用 CUDA: NVIDIA GeForce RTX 4050 Laptop GPU
nptdms = available


In [10]:
# =========================
# Config
# =========================
RAW_DATA_ROOT = Path(r'G:\20260323_ZZ_pccp\FIP\24-900-1800\test')

WINDOW_DURATION_S = 0.02
WINDOW_OVERLAP = 0.50
assert 0.0 <= WINDOW_OVERLAP < 1.0

NPZ_PER_CSV = 100

# Window-level parallel settings
WINDOW_WORKERS = 6
WINDOW_BATCH_SIZE = 256

# TDMS input hints
TDMS_GROUP_NAME: str | None = None
TDMS_CHANNEL_NAME: str | None = None
TDMS_FALLBACK_SAMPLE_RATE_HZ: float | None = None

SELECTED_FEATURES = [
    'b_1k_10k__SC_mean',
    'b_1k_10k__C_f',
    'b_1k_100k__SC_mean',
    'b_1k_10k__C_h',
    'b_40k_60k__SC_mean',
]

BANDS = {
    'b_1k_100k': (1_000.0, 100_000.0),
    'b_1k_10k': (1_000.0, 10_000.0),
    'b_40k_60k': (40_000.0, 60_000.0),
}

# Whole-signal preprocess bandpass
PREPROC_BAND = (1_000.0, 95_000.0)

OUTPUT_ROOT = workspace / 'outputs' / 'realdata_feature_dataset_20260519_v3'
FEATURE_CSV_PREFIX = 'fip24afternoon_window_features_v3'
LOG_CSV_PREFIX = 'fip24afternoon_window_log_v3'
RUNTIME_LOG_NAME = 'fip24afternoon_runtime_v3.log'
PROCESSED_LIST_NAME = 'processed_source_files_v3.txt'

MAX_FILES: int | None = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'RAW_DATA_ROOT={RAW_DATA_ROOT}')
print(f'OUTPUT_ROOT={OUTPUT_ROOT}')
print(f'WINDOW_WORKERS={WINDOW_WORKERS}, WINDOW_BATCH_SIZE={WINDOW_BATCH_SIZE}')



RAW_DATA_ROOT=G:\20260323_ZZ_pccp\FIP\24-900-1800\test
OUTPUT_ROOT=e:\codes\ZZ-BK\outputs\realdata_feature_dataset_20260519_v3
WINDOW_WORKERS=6, WINDOW_BATCH_SIZE=256


In [11]:
# =========================
# Helpers
# =========================

def build_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger('realdata_feature_dataset_v3')
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')

    fh = logging.FileHandler(log_path, encoding='utf-8')
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger


def _safe_band(low: float, high: float, nyq: float) -> tuple[float, float]:
    low = max(1.0, min(low, nyq * 0.98))
    high = max(low + 1.0, min(high, nyq * 0.995))
    return (float(low), float(high))


def _scalar_text(value: object) -> str:
    if value is None:
        return ''
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='ignore').strip()
    if isinstance(value, np.generic):
        value = value.item()
    if hasattr(value, 'tolist') and not isinstance(value, str):
        try:
            value = value.tolist()
        except Exception:
            pass
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return _scalar_text(value[0])
    return str(value).strip()


def _first_property(props: dict[str, object], names: tuple[str, ...]) -> object | None:
    normalized = {str(k).lower(): v for k, v in props.items()}
    for name in names:
        if name.lower() in normalized:
            return normalized[name.lower()]
    return None


def _coerce_float(value: object | None) -> float | None:
    if value is None:
        return None
    try:
        arr = np.asarray(value)
        if arr.shape == ():
            return float(arr.item())
        if arr.size == 1:
            return float(arr.reshape(()).item())
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return None


def _infer_sample_rate_from_filename(path: Path) -> float | None:
    import re
    m = re.search(r'(?<!\d)(\d+(?:\.\d+)?)\s*([kKmM])(?![a-zA-Z])', path.stem)
    if not m:
        return None
    val = float(m.group(1))
    unit = m.group(2).lower()
    if unit == 'k':
        return val * 1_000.0
    if unit == 'm':
        return val * 1_000_000.0
    return None


def build_params_for_band(band: tuple[float, float], sample_rate: float):
    low, high = band
    nyq = sample_rate / 2.0
    low, high = _safe_band(low, high, nyq)
    span = max(high - low, 10.0)

    low_band = _safe_band(low, low + 0.30 * span, nyq)
    mid_band = _safe_band(low + 0.30 * span, low + 0.60 * span, nyq)
    high1_band = _safe_band(low + 0.50 * span, low + 0.80 * span, nyq)
    high2_band = _safe_band(low + 0.60 * span, high, nyq)
    harmonic_band = _safe_band(low + 0.50 * span, high, nyq)
    ridge_main = _safe_band(low, low + 0.65 * span, nyq)
    ridge_h2 = _safe_band(max(low * 2.0, low + 0.20 * span), min(high * 2.0, nyq * 0.995), nyq)

    return replace(
        DEFAULT_FEATURE_PARAMS,
        highpass_hz=1_000.0,
        main_band_hz=(low, high),
        low_band_hz=low_band,
        mid_band_hz=mid_band,
        high1_band_hz=high1_band,
        high2_band_hz=high2_band,
        harmonic_band_hz=harmonic_band,
        ridge_main_search_hz=ridge_main,
        ridge_h2_search_hz=ridge_h2,
        n_jobs=1,
    )


def _spectral_centroid_mean(freqs: np.ndarray, power: np.ndarray, eps: float) -> float:
    if power.size == 0:
        return 0.0
    numerator = np.sum(freqs[:, None] * power, axis=0)
    denominator = np.sum(power, axis=0) + eps
    sc = numerator / denominator
    return float(np.mean(sc)) if sc.size else 0.0


def _c_f_from_context(context) -> float:
    eps = context.params.eps
    if len(context.ridge_f1) <= 2:
        return 0.0
    curvature = np.gradient(
        np.gradient(context.ridge_f1, context.stft_times + eps),
        context.stft_times + eps,
    )
    return float(np.mean(np.abs(curvature) / (np.mean(np.abs(context.ridge_f1)) + eps))) if curvature.size else 0.0


def _c_h_from_context(context) -> float:
    active = context.ridge_f1 > 0.0
    if not np.any(active):
        return 0.0
    diff_h2 = np.abs(context.ridge_f2 - 2.0 * context.ridge_f1)
    return float(np.mean(diff_h2[active]))


def compute_5_features_for_window(window_signal: np.ndarray, sample_rate: float, params_map: dict[str, object]) -> dict[str, float]:
    rec = FeatureRecord(
        sample_id='w',
        sample_name='w',
        sample_type='raw',
        sample_type_code=0,
        path=Path('.'),
        signal=np.asarray(window_signal, dtype=float),
        sample_rate=float(sample_rate),
        metadata={},
    )

    out: dict[str, float] = {}

    ctx_1k10k = build_context(rec, params_map['b_1k_10k'])
    out['b_1k_10k__SC_mean'] = _spectral_centroid_mean(ctx_1k10k.stft_freqs, ctx_1k10k.stft_power, ctx_1k10k.params.eps)
    out['b_1k_10k__C_f'] = _c_f_from_context(ctx_1k10k)
    out['b_1k_10k__C_h'] = _c_h_from_context(ctx_1k10k)

    ctx_1k100k = build_context(rec, params_map['b_1k_100k'])
    out['b_1k_100k__SC_mean'] = _spectral_centroid_mean(ctx_1k100k.stft_freqs, ctx_1k100k.stft_power, ctx_1k100k.params.eps)

    ctx_40k60k = build_context(rec, params_map['b_40k_60k'])
    out['b_40k_60k__SC_mean'] = _spectral_centroid_mean(ctx_40k60k.stft_freqs, ctx_40k60k.stft_power, ctx_40k60k.params.eps)

    return out


def parse_starttime(starttime_raw: str) -> datetime | None:
    if not starttime_raw:
        return None
    fmts = [
        '%Y%m%dT%H%M%S.%f', '%Y%m%dT%H%M%S',
        '%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S',
        '%Y-%m-%dT%H:%M:%S.%f', '%Y-%m-%dT%H:%M:%S',
    ]
    for fmt in fmts:
        try:
            return datetime.strptime(starttime_raw, fmt)
        except Exception:
            pass
    try:
        return datetime.fromisoformat(starttime_raw.replace('Z', '+00:00'))
    except Exception:
        return None


def list_window_ranges(n_samples: int, sample_rate: float, window_duration_s: float, overlap: float) -> list[tuple[int, int, int, int, int]]:
    win = int(round(window_duration_s * sample_rate))
    if win <= 0:
        raise ValueError('window_samples must be positive')
    if n_samples < win:
        return []
    step = max(1, int(round(win * (1.0 - overlap))))
    out = []
    idx = 0
    wid = 0
    while idx + win <= n_samples:
        out.append((wid, idx, idx + win, win, step))
        idx += step
        wid += 1
    return out


def _load_npz_source(path: Path) -> dict[str, object]:
    with np.load(path, allow_pickle=True) as data:
        signal_values = np.asarray(data['phase_data'], dtype=float)
        sample_rate = float(np.asarray(data['sample_rate']).item())
        starttime_raw = _scalar_text(data.get('starttime', '')) if 'starttime' in data else ''
        arrival_time_raw = _scalar_text(data.get('arrival_time', '')) if 'arrival_time' in data else ''
        sample_type = _scalar_text(data.get('type', path.parent.name)) if 'type' in data else path.parent.name
    return {
        'source_format': 'npz',
        'signal_values': signal_values,
        'sample_rate': sample_rate,
        'starttime_raw': starttime_raw,
        'arrival_time_raw': arrival_time_raw,
        'sample_type': sample_type,
        'source_group_name': '',
        'source_channel_name': '',
        'source_detail': '',
    }


def _select_tdms_channel(tdms_file):
    if TDMS_GROUP_NAME and TDMS_CHANNEL_NAME:
        for g in tdms_file.groups():
            if str(g.name).lower() == TDMS_GROUP_NAME.lower():
                for c in g.channels():
                    if str(c.name).lower() == TDMS_CHANNEL_NAME.lower():
                        return g, c
        raise ValueError(f'Cannot find TDMS group/channel: {TDMS_GROUP_NAME}/{TDMS_CHANNEL_NAME}')

    if TDMS_CHANNEL_NAME:
        for g in tdms_file.groups():
            for c in g.channels():
                if str(c.name).lower() == TDMS_CHANNEL_NAME.lower():
                    return g, c

    pref = {'phase_data', 'signal', 'data', 'values', 'channel0', 'ch0'}
    for g in tdms_file.groups():
        for c in g.channels():
            if str(c.name).lower() in pref:
                return g, c

    best = None
    best_len = -1
    for g in tdms_file.groups():
        for c in g.channels():
            try:
                arr = np.asarray(c[:])
                if arr.size == 0:
                    continue
                if not np.issubdtype(arr.dtype, np.number):
                    arr = arr.astype(float)
            except Exception:
                continue
            if arr.size > best_len:
                best_len = arr.size
                best = (g, c)
    if best is None:
        raise ValueError('No usable numeric channel found in TDMS')
    return best


def _load_tdms_source(path: Path) -> dict[str, object]:
    if TdmsFile is None:
        raise ImportError('nptdms is required for .tdms files. Install with: pip install nptdms')

    td = TdmsFile.read(path)
    g, c = _select_tdms_channel(td)

    signal_values = np.asarray(c[:], dtype=float)
    props = {}
    props.update(getattr(td, 'properties', {}) or {})
    props.update(getattr(g, 'properties', {}) or {})
    props.update(getattr(c, 'properties', {}) or {})

    sample_rate = _coerce_float(_first_property(props, ('sample_rate', 'sample_rate_hz', 'sampling_rate', 'sampling_rate_hz')))
    if sample_rate is None:
        wf_inc = _coerce_float(_first_property(props, ('wf_increment',)))
        if wf_inc and wf_inc > 0:
            sample_rate = 1.0 / wf_inc

    if sample_rate is None or sample_rate <= 0:
        sample_rate = _infer_sample_rate_from_filename(path)

    if (sample_rate is None or sample_rate <= 0) and TDMS_FALLBACK_SAMPLE_RATE_HZ is not None:
        sample_rate = float(TDMS_FALLBACK_SAMPLE_RATE_HZ)

    if sample_rate is None or sample_rate <= 0:
        raise ValueError(
            f'Cannot infer sample rate from TDMS file: {path}. '
            'Provide TDMS_FALLBACK_SAMPLE_RATE_HZ or include rate text like 500K in filename.'
        )

    starttime_raw = _scalar_text(_first_property(props, ('starttime', 'start_time', 'wf_start_time', 'wf_starttime')))
    arrival_time_raw = _scalar_text(_first_property(props, ('arrival_time', 'arrivaltime', 'arrival_time_text')))
    sample_type = _scalar_text(_first_property(props, ('type', 'sample_type', 'sampletype'))) or path.parent.name

    return {
        'source_format': 'tdms',
        'signal_values': signal_values,
        'sample_rate': float(sample_rate),
        'starttime_raw': starttime_raw,
        'arrival_time_raw': arrival_time_raw,
        'sample_type': sample_type,
        'source_group_name': str(g.name),
        'source_channel_name': str(c.name),
        'source_detail': f'{g.name}/{c.name}',
    }


def load_source_file(path: Path) -> dict[str, object]:
    suf = path.suffix.lower()
    if suf == '.npz':
        return _load_npz_source(path)
    if suf == '.tdms':
        return _load_tdms_source(path)
    raise ValueError(f'Unsupported file type: {path.suffix}')



In [12]:
# =========================
# Main pipeline
# =========================
runtime_log_path = OUTPUT_ROOT / RUNTIME_LOG_NAME
processed_list_path = OUTPUT_ROOT / PROCESSED_LIST_NAME
logger = build_logger(runtime_log_path)

source_files = sorted(
    p for p in RAW_DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in {'.npz', '.tdms'}
)
if MAX_FILES is not None:
    source_files = source_files[:MAX_FILES]
if not source_files:
    raise FileNotFoundError(f'No npz/tdms files found under: {RAW_DATA_ROOT}')

processed_set: set[str] = set()
if processed_list_path.exists():
    processed_set = {ln.strip() for ln in processed_list_path.read_text(encoding='utf-8').splitlines() if ln.strip()}

logger.info('Found %d source files', len(source_files))
logger.info('Already processed: %d', len(processed_set))
logger.info('Window config: duration=%.6fs overlap=%.2f', WINDOW_DURATION_S, WINDOW_OVERLAP)
logger.info('Selected features: %s', ', '.join(SELECTED_FEATURES))
logger.info('NPZ_PER_CSV = %d', NPZ_PER_CSV)
logger.info('WINDOW_WORKERS = %d, WINDOW_BATCH_SIZE = %d', WINDOW_WORKERS, WINDOW_BATCH_SIZE)

processed_now = 0
window_total = 0


def chunk_paths(chunk_index: int) -> tuple[Path, Path]:
    suffix = f'part_{chunk_index:04d}.csv'
    return (
        OUTPUT_ROOT / f'{FEATURE_CSV_PREFIX}_{suffix}',
        OUTPUT_ROOT / f'{LOG_CSV_PREFIX}_{suffix}',
    )

for file_idx, fp in enumerate(tqdm(source_files, desc='Files'), start=1):
    fp_str = str(fp)
    if fp_str in processed_set:
        continue

    chunk_index = (file_idx - 1) // NPZ_PER_CSV + 1
    feature_csv_path, log_csv_path = chunk_paths(chunk_index)

    src = load_source_file(fp)
    raw_signal = np.asarray(src['signal_values'], dtype=float)
    sample_rate = float(src['sample_rate'])

    # Preprocess once per source file
    centered = raw_signal - float(np.mean(raw_signal))
    signal_pre = butter_filter(centered, sample_rate=sample_rate, band_hz=PREPROC_BAND, order=4)

    starttime_raw = str(src['starttime_raw'])
    arrival_time_raw = str(src['arrival_time_raw'])
    sample_type = str(src['sample_type'])
    source_format = str(src['source_format'])
    source_group_name = str(src.get('source_group_name', ''))
    source_channel_name = str(src.get('source_channel_name', ''))
    source_detail = str(src.get('source_detail', ''))

    start_dt = parse_starttime(starttime_raw)
    n_samples = len(signal_pre)
    duration_s = n_samples / sample_rate if sample_rate > 0 else np.nan

    params_map = {k: build_params_for_band(v, sample_rate) for k, v in BANDS.items()}
    windows = list_window_ranges(n_samples, sample_rate, WINDOW_DURATION_S, WINDOW_OVERLAP)

    rows_features: list[dict[str, object]] = []
    rows_log: list[dict[str, object]] = []

    def process_one_window(win_tuple):
        win_id, i0, i1, win_len, step_len = win_tuple
        win_signal = signal_pre[i0:i1]
        fvals = compute_5_features_for_window(win_signal, sample_rate, params_map)

        base = {
            'source_file_name': fp.name,
            'source_file_path': fp_str,
            'source_format': source_format,
            'source_group_name': source_group_name,
            'source_channel_name': source_channel_name,
            'source_detail': source_detail,
            'window_id': int(win_id),
            'window_start_index': int(i0),
            'window_end_index': int(i1),
            'window_length_samples': int(win_len),
            'window_step_samples': int(step_len),
            'window_duration_s': float(win_len / sample_rate),
            'window_start_offset_s': float(i0 / sample_rate),
            'sample_rate_hz': float(sample_rate),
            'source_n_samples': int(n_samples),
            'source_duration_s': float(duration_s),
            'starttime_raw': starttime_raw,
            'arrival_time_raw': arrival_time_raw,
            'sample_type': sample_type,
            'csv_chunk_index': int(chunk_index),
        }
        if start_dt is not None:
            base['window_start_datetime'] = (start_dt + timedelta(seconds=float(i0 / sample_rate))).strftime('%Y-%m-%d %H:%M:%S.%f')
        else:
            base['window_start_datetime'] = ''

        feat_row = dict(base)
        for fn in SELECTED_FEATURES:
            feat_row[fn] = float(fvals.get(fn, np.nan))

        log_row = dict(base)
        log_row['missing_selected_features'] = ','.join([f for f in SELECTED_FEATURES if f not in fvals])
        return feat_row, log_row

    # Window-level parallel in batches
    max_workers = max(1, int(WINDOW_WORKERS))
    for b0 in range(0, len(windows), WINDOW_BATCH_SIZE):
        chunk = windows[b0:b0 + WINDOW_BATCH_SIZE]
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(process_one_window, w) for w in chunk]
            for fut in as_completed(futures):
                feat_row, log_row = fut.result()
                rows_features.append(feat_row)
                rows_log.append(log_row)

    # keep deterministic order by window_id
    rows_features.sort(key=lambda x: int(x['window_id']))
    rows_log.sort(key=lambda x: int(x['window_id']))

    df_features = pd.DataFrame(rows_features)
    df_log = pd.DataFrame(rows_log)

    feature_header = (not feature_csv_path.exists()) or (feature_csv_path.stat().st_size == 0)
    log_header = (not log_csv_path.exists()) or (log_csv_path.stat().st_size == 0)
    df_features.to_csv(feature_csv_path, mode='a', header=feature_header, index=False, encoding='utf-8-sig')
    df_log.to_csv(log_csv_path, mode='a', header=log_header, index=False, encoding='utf-8-sig')

    with processed_list_path.open('a', encoding='utf-8') as f:
        f.write(fp_str + '\n')
    processed_set.add(fp_str)

    processed_now += 1
    window_total += len(df_features)
    logger.info('Processed file=%s, format=%s, sample_rate=%.1fHz, n_samples=%d, windows=%d, chunk=%d', fp.name, source_format, sample_rate, n_samples, len(df_features), chunk_index)

logger.info('Run finished. Newly processed files=%d, total windows in this run=%d', processed_now, window_total)
logger.info('Processed list: %s', processed_list_path)
print('Done')
print(f'newly_processed_files={processed_now}')
print(f'total_windows_this_run={window_total}')


[2026-05-19 16:34:11,784] INFO: Found 330 source files
[2026-05-19 16:34:11,785] INFO: Already processed: 0
[2026-05-19 16:34:11,785] INFO: Window config: duration=0.020000s overlap=0.50
[2026-05-19 16:34:11,786] INFO: Selected features: b_1k_10k__SC_mean, b_1k_10k__C_f, b_1k_100k__SC_mean, b_1k_10k__C_h, b_40k_60k__SC_mean
[2026-05-19 16:34:11,786] INFO: NPZ_PER_CSV = 100
[2026-05-19 16:34:11,786] INFO: WINDOW_WORKERS = 6, WINDOW_BATCH_SIZE = 256


Files:   0%|          | 0/330 [00:00<?, ?it/s]

[2026-05-19 16:35:03,228] INFO: Processed file=0000355-500K-20260324T143951.852.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


Files:   0%|          | 1/330 [00:51<4:42:03, 51.44s/it]

[2026-05-19 16:35:53,656] INFO: Processed file=0000356-500K-20260324T144001.853.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


...

[2026-05-19 17:01:53,419] INFO: Processed file=0000386-500K-20260324T144501.936.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


Files:  10%|▉         | 32/330 [27:41<4:35:49, 55.54s/it]

[2026-05-19 17:02:44,513] INFO: Processed file=0000387-500K-20260324T144511.858.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


Files:  10%|█         | 33/330 [28:46<4:18:58, 52.32s/it]


KeyboardInterrupt: 

In [ ]:
# Quick check
feature_chunks = sorted(OUTPUT_ROOT.glob(f'{FEATURE_CSV_PREFIX}_part_*.csv'))
log_chunks = sorted(OUTPUT_ROOT.glob(f'{LOG_CSV_PREFIX}_part_*.csv'))
print('feature chunk count =', len(feature_chunks))
print('log chunk count =', len(log_chunks))
if feature_chunks:
    print('last feature chunk =', feature_chunks[-1])
if log_chunks:
    print('last log chunk =', log_chunks[-1])
